In [ ]:
import gc
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

SEED = 42
ALPHA = 0.50
BATCH_SIZE = 8
MAX_EPOCHS = 100
LEARNING_RATE = 5e-5
EARLY_STOP_PATIENCE = 10
MIN_DELTA = 1e-6

ABLATION_MODES = (
    "no_epsilon",
    "fixed_theta",
    "fixed_theta_no_epsilon",
)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("results") / f"corrected_hpb_idb_component_ablation_seed{SEED}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = OUTPUT_DIR / f"idb_component_ablation_metrics_seed{SEED}.csv"

def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED)
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"IDB component ablation modes: {ABLATION_MODES}")


In [ ]:
def load_data():
    data_paths = {
        "combination_response": DATA_DIR / "processed_combination_response_r070_clean_with_qc.csv",
        "drug_features": DATA_DIR / "Graphlet_features_6_standardized.csv",
        "cell_features": DATA_DIR / "cell_features_977d.csv",
    }

    missing_files = [path for path in data_paths.values() if not path.is_file()]
    if missing_files:
        missing_text = "\n".join(f"  - {path}" for path in missing_files)
        raise FileNotFoundError(
            "Required data files were not found.\n"
            f"Please check DATA_DIR: {DATA_DIR}\n"
            f"Missing files:\n{missing_text}"
        )

    combo_df = pd.read_csv(data_paths["combination_response"], encoding="utf-8-sig")
    combo_df = combo_df.rename(
        columns={
            "target": "X/X0",
            "drugA_conc": "drugA Conc (µM)",
            "drugB_conc": "drugB Conc (µM)",
        }
    )
    combo_df["drugA_name"] = combo_df["drugA_name"].astype(str).str.strip().str.upper()
    combo_df["drugB_name"] = combo_df["drugB_name"].astype(str).str.strip().str.upper()
    combo_df["cell_line"] = combo_df["cell_line"].astype(str).str.strip()

    drug_features = pd.read_csv(data_paths["drug_features"], encoding="utf-8-sig")
    drug_features["name"] = drug_features["name"].astype(str).str.strip().str.upper()
    drug_features.set_index("name", inplace=True)

    cell_features = pd.read_csv(data_paths["cell_features"], encoding="utf-8-sig")
    cell_features["Cell_Line"] = cell_features["Cell_Line"].astype(str).str.strip()
    cell_features.set_index("Cell_Line", inplace=True)
    return combo_df, drug_features, cell_features

def preprocess_data(combo_df, drug_features, cell_features):
    valid_mask = (
        combo_df["drugA_name"].isin(drug_features.index)
        & combo_df["drugB_name"].isin(drug_features.index)
        & combo_df["cell_line"].isin(cell_features.index)
    )
    skipped_count = len(combo_df) - int(valid_mask.sum())
    print(f"Skipped {skipped_count} samples due to missing features")

    data_df = combo_df.loc[
        valid_mask,
        [
            "drugA_name",
            "drugB_name",
            "cell_line",
            "drugA Conc (µM)",
            "drugB Conc (µM)",
            "X/X0",
            "single_resp_1",
            "single_resp_2",
        ],
    ].copy()
    data_df = data_df.rename(
        columns={
            "drugA Conc (µM)": "drugA_conc",
            "drugB Conc (µM)": "drugB_conc",
            "X/X0": "target",
        }
    )
    return data_df.reset_index(drop=True)

def add_group_id(data_df):
    grouped_df = data_df.copy()
    canonical_pairs = np.sort(
        grouped_df[["drugA_name", "drugB_name"]].astype(str).to_numpy(), axis=1
    )
    grouped_df["group_id"] = (
        canonical_pairs[:, 0]
        + "||"
        + canonical_pairs[:, 1]
        + "||"
        + grouped_df["cell_line"].astype(str).to_numpy()
    )
    return grouped_df

def grouped_train_val_test_split(
    data_df, train_size=0.70, val_size=0.10, test_size=0.20, random_state=SEED
):
    if not np.isclose(train_size + val_size + test_size, 1.0):
        raise ValueError("train_size + val_size + test_size must equal 1.0")

    grouped_df = add_group_id(data_df)
    outer_split = GroupShuffleSplit(
        n_splits=1, test_size=test_size, random_state=random_state
    )
    train_val_idx, test_idx = next(
        outer_split.split(grouped_df, groups=grouped_df["group_id"])
    )
    train_val_df = grouped_df.iloc[train_val_idx].copy()
    test_df = grouped_df.iloc[test_idx].copy()

    relative_val_size = val_size / (train_size + val_size)
    inner_split = GroupShuffleSplit(
        n_splits=1, test_size=relative_val_size, random_state=random_state + 1
    )
    train_idx, val_idx = next(
        inner_split.split(train_val_df, groups=train_val_df["group_id"])
    )
    train_df = train_val_df.iloc[train_idx].copy().reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].copy().reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    train_groups = set(train_df["group_id"])
    val_groups = set(val_df["group_id"])
    test_groups = set(test_df["group_id"])
    overlap_counts = {
        "train_validation": len(train_groups & val_groups),
        "train_test": len(train_groups & test_groups),
        "validation_test": len(val_groups & test_groups),
    }
    assert all(value == 0 for value in overlap_counts.values()), overlap_counts
    assert len(train_df) + len(val_df) + len(test_df) == len(grouped_df)
    assert train_groups | val_groups | test_groups == set(grouped_df["group_id"])

    split_summary = pd.DataFrame(
        {
            "split": ["Train", "Validation", "Test"],
            "samples": [len(train_df), len(val_df), len(test_df)],
            "groups": [len(train_groups), len(val_groups), len(test_groups)],
        }
    )
    split_summary["sample_percent"] = (
        split_summary["samples"] / len(grouped_df) * 100
    ).round(4)
    split_summary["group_percent"] = (
        split_summary["groups"] / grouped_df["group_id"].nunique() * 100
    ).round(4)

    print("\nGroup-aware split summary:")
    display(split_summary)
    print(f"Group-overlap audit: {overlap_counts} | PASSED")
    return train_df, val_df, test_df, split_summary


In [ ]:
class DrugCombinationDataset(Dataset):
    def __init__(self, data, drug_features, cell_features):
        self.data = data.reset_index(drop=True)
        self.drug_features = {
            name: row.values[2:].astype(np.float32)
            for name, row in drug_features.iterrows()
        }
        self.cell_features = {
            name: row.values.astype(np.float32)
            for name, row in cell_features.iterrows()
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data.iloc[idx]
        drugA_feat = self.drug_features[sample["drugA_name"]]
        drugB_feat = self.drug_features[sample["drugB_name"]]
        cell_feat = self.cell_features[sample["cell_line"]]
        combo_feat = np.concatenate([drugA_feat, drugB_feat, cell_feat]).astype(np.float32)
        return {
            "drugA_feat": torch.from_numpy(drugA_feat),
            "drugB_feat": torch.from_numpy(drugB_feat),
            "cell_feat": torch.from_numpy(cell_feat),
            "combo_feat": torch.from_numpy(combo_feat),
            "target": torch.tensor(sample["target"], dtype=torch.float32),
            "drugA_conc": torch.tensor(sample["drugA_conc"], dtype=torch.float32),
            "drugB_conc": torch.tensor(sample["drugB_conc"], dtype=torch.float32),
            "single_resp_1": torch.tensor(sample["single_resp_1"], dtype=torch.float32),
            "single_resp_2": torch.tensor(sample["single_resp_2"], dtype=torch.float32),
            "index": idx,
        }

class DrugCombinationModel(nn.Module):
    def __init__(self, drug_feat_dim=1000, cell_feat_dim=977):
        super(DrugCombinationModel, self).__init__()

        self.drug_encoder = nn.Sequential(
            nn.Linear(drug_feat_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )

        self.cell_encoder = nn.Sequential(
            nn.Linear(cell_feat_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )

        self.theta1_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

        self.theta2_net = nn.Sequential(
            nn.Linear(256 + 256 + 1, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

        self.epsilon_net = nn.Sequential(
            nn.Linear(256 * 3 + 256 + 2, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

        self.dose_encoder = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
        )

        hpb_input_dim = 256 + 256 + 256 + 32 + 32
        self.predictor_direct = nn.Sequential(
            nn.Linear(hpb_input_dim, 2048),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        drugA_feat,
        drugB_feat,
        cell_feat,
        combo_feat,
        drugA_conc,
        drugB_conc,
    ):
        encoded_drugA = self.drug_encoder(drugA_feat)
        encoded_drugB = self.drug_encoder(drugB_feat)
        encoded_cell = self.cell_encoder(cell_feat)

        theta1_input = torch.cat(
            [encoded_drugA, encoded_cell, drugA_conc.unsqueeze(1)], dim=1
        )
        theta2_input = torch.cat(
            [encoded_drugB, encoded_cell, drugB_conc.unsqueeze(1)], dim=1
        )
        theta1_raw = self.theta1_net(theta1_input)
        theta2_raw = self.theta2_net(theta2_input)

        drug_pair_sum = encoded_drugA + encoded_drugB
        drug_pair_product = encoded_drugA * encoded_drugB
        drug_pair_abs_difference = torch.abs(encoded_drugA - encoded_drugB)
        epsilon_input = torch.cat(
            [
                drug_pair_sum,
                drug_pair_product,
                drug_pair_abs_difference,
                encoded_cell,
                drugA_conc.unsqueeze(1),
                drugB_conc.unsqueeze(1),
            ],
            dim=1,
        )
        epsilon = self.epsilon_net(epsilon_input)

        theta12_sum = theta1_raw + theta2_raw
        theta1 = theta1_raw / theta12_sum
        theta2 = theta2_raw / theta12_sum

        encoded_doseA = self.dose_encoder(drugA_conc.unsqueeze(1))
        encoded_doseB = self.dose_encoder(drugB_conc.unsqueeze(1))

        combined_features = torch.cat(
            [
                encoded_drugA,
                encoded_drugB,
                encoded_cell,
                encoded_doseA,
                encoded_doseB,
            ],
            dim=1,
        )
        p_direct = self.predictor_direct(combined_features)

        return (
            theta1.squeeze(-1),
            theta2.squeeze(-1),
            epsilon.squeeze(-1),
            p_direct.squeeze(-1),
        )


In [ ]:
def compose_predictions(
    theta1, theta2, epsilon, p_hpb, single_resp_1, single_resp_2, ablation_mode
):
    if ablation_mode == "full":
        effective_theta1 = theta1
        effective_theta2 = theta2
        effective_epsilon = epsilon
    elif ablation_mode == "no_epsilon":
        effective_theta1 = theta1
        effective_theta2 = theta2
        effective_epsilon = torch.zeros_like(epsilon)
    elif ablation_mode == "fixed_theta":
        effective_theta1 = torch.full_like(theta1, 0.5)
        effective_theta2 = torch.full_like(theta2, 0.5)
        effective_epsilon = epsilon
    elif ablation_mode == "fixed_theta_no_epsilon":
        effective_theta1 = torch.full_like(theta1, 0.5)
        effective_theta2 = torch.full_like(theta2, 0.5)
        effective_epsilon = torch.zeros_like(epsilon)
    else:
        raise ValueError(f"Unknown ablation mode: {ablation_mode}")

    p_idb = (
        effective_theta1 * single_resp_1
        + effective_theta2 * single_resp_2
        + effective_epsilon
    )
    p_final = ALPHA * p_idb + (1.0 - ALPHA) * p_hpb
    return (
        p_idb,
        p_final,
        effective_theta1,
        effective_theta2,
        effective_epsilon,
    )

def concordance_index(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    finite = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[finite], y_pred[finite]
    if len(y_true) < 2:
        return np.nan

    order = np.argsort(y_true, kind="mergesort")
    y_true, y_pred = y_true[order], y_pred[order]
    _, pred_rank = np.unique(y_pred, return_inverse=True)
    pred_rank = pred_rank + 1
    tree = np.zeros(pred_rank.max() + 1, dtype=np.int64)

    def bit_add(index):
        while index < len(tree):
            tree[index] += 1
            index += index & -index

    def bit_sum(index):
        total = 0
        while index > 0:
            total += tree[index]
            index -= index & -index
        return total

    concordant = 0.0
    comparable = 0
    previous_count = 0
    start = 0
    while start < len(y_true):
        end = start + 1
        while end < len(y_true) and y_true[end] == y_true[start]:
            end += 1
        for rank in pred_rank[start:end]:
            lower = bit_sum(rank - 1)
            equal = bit_sum(rank) - lower
            concordant += lower + 0.5 * equal
            comparable += previous_count
        for rank in pred_rank[start:end]:
            bit_add(rank)
        previous_count += end - start
        start = end
    return concordant / comparable if comparable else np.nan

def calculate_metrics(targets, predictions):
    targets = np.asarray(targets, dtype=float)
    predictions = np.asarray(predictions, dtype=float)
    mse = mean_squared_error(targets, predictions)
    return {
        "mse": mse,
        "rmse": float(np.sqrt(mse)),
        "mae": mean_absolute_error(targets, predictions),
        "pcc": pearsonr(targets, predictions)[0],
        "r2": r2_score(targets, predictions),
        "c_index": concordance_index(targets, predictions),
    }

def forward_batch(model, batch, ablation_mode):
    drugA_feat = batch["drugA_feat"].to(device)
    drugB_feat = batch["drugB_feat"].to(device)
    cell_feat = batch["cell_feat"].to(device)
    combo_feat = batch["combo_feat"].to(device)
    drugA_conc = batch["drugA_conc"].to(device)
    drugB_conc = batch["drugB_conc"].to(device)
    single_resp_1 = batch["single_resp_1"].to(device)
    single_resp_2 = batch["single_resp_2"].to(device)

    theta1, theta2, epsilon, p_hpb = model(
        drugA_feat,
        drugB_feat,
        cell_feat,
        combo_feat,
        drugA_conc,
        drugB_conc,
    )
    p_idb, p_final, eff_theta1, eff_theta2, eff_epsilon = compose_predictions(
        theta1,
        theta2,
        epsilon,
        p_hpb,
        single_resp_1,
        single_resp_2,
        ablation_mode,
    )
    return {
        "target": batch["target"].to(device),
        "theta1": theta1,
        "theta2": theta2,
        "epsilon": epsilon,
        "effective_theta1": eff_theta1,
        "effective_theta2": eff_theta2,
        "effective_epsilon": eff_epsilon,
        "p_idb": p_idb,
        "p_hpb": p_hpb,
        "p_final": p_final,
    }


In [ ]:
def evaluate_model(model, data_loader, data_df, ablation_mode, collect_results=False):
    model.eval()
    all_targets = []
    all_predictions = []
    result_frames = []

    with torch.no_grad():
        for batch in data_loader:
            outputs = forward_batch(model, batch, ablation_mode)
            targets = outputs["target"].detach().cpu().numpy().reshape(-1)
            predictions = outputs["p_final"].detach().cpu().numpy().reshape(-1)
            all_targets.extend(targets)
            all_predictions.extend(predictions)

            if collect_results:
                indices = batch["index"].numpy()
                frame = data_df.iloc[indices][
                    [
                        "drugA_name",
                        "drugB_name",
                        "cell_line",
                        "drugA_conc",
                        "drugB_conc",
                        "single_resp_1",
                        "single_resp_2",
                        "group_id",
                    ]
                ].copy()
                frame.insert(0, "seed", SEED)
                frame.insert(1, "ablation_mode", ablation_mode)
                frame["target"] = targets
                frame["prediction"] = predictions
                frame["idb_prediction"] = outputs["p_idb"].cpu().numpy().reshape(-1)
                frame["hpb_prediction"] = outputs["p_hpb"].cpu().numpy().reshape(-1)
                frame["effective_theta1"] = outputs["effective_theta1"].cpu().numpy().reshape(-1)
                frame["effective_theta2"] = outputs["effective_theta2"].cpu().numpy().reshape(-1)
                frame["effective_epsilon"] = outputs["effective_epsilon"].cpu().numpy().reshape(-1)
                result_frames.append(frame)

    metrics = calculate_metrics(all_targets, all_predictions)
    results_df = pd.concat(result_frames, ignore_index=True) if result_frames else None
    return metrics, results_df

def train_one_mode(model, train_loader, val_loader, val_df, ablation_mode):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-7
    )

    checkpoint_path = OUTPUT_DIR / f"{ablation_mode}_best_model_seed{SEED}.pth"
    history_path = OUTPUT_DIR / f"{ablation_mode}_training_history_seed{SEED}.csv"
    best_val_loss = float("inf")
    best_epoch = 0
    no_improve_count = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            outputs = forward_batch(model, batch, ablation_mode)
            loss = criterion(outputs["p_final"], outputs["target"])
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(outputs["target"])

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics, _ = evaluate_model(
            model, val_loader, val_df, ablation_mode, collect_results=False
        )
        val_loss = val_metrics["mse"]
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]
        history.append(
            {
                "epoch": epoch,
                "train_mse": train_loss,
                "validation_mse": val_loss,
                "validation_rmse": val_metrics["rmse"],
                "validation_mae": val_metrics["mae"],
                "validation_pcc": val_metrics["pcc"],
                "validation_r2": val_metrics["r2"],
                "validation_c_index": val_metrics["c_index"],
                "learning_rate": current_lr,
            }
        )
        pd.DataFrame(history).to_csv(history_path, index=False, encoding="utf-8-sig")

        print(
            f"{ablation_mode} | Epoch {epoch}/{MAX_EPOCHS} | "
            f"Train {train_loss:.6f} | Val {val_loss:.6f} | LR {current_lr:.8f}"
        )
        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_epoch = epoch
            no_improve_count = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            no_improve_count += 1

        if no_improve_count >= EARLY_STOP_PATIENCE:
            print(f"{ablation_mode} | Early stopping at epoch {epoch}")
            break

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    history_df = pd.DataFrame(history)
    plt.figure(figsize=(8, 4.5))
    plt.plot(history_df["epoch"], history_df["train_mse"], label="Train MSE")
    plt.plot(history_df["epoch"], history_df["validation_mse"], label="Validation MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(f"IDB component ablation: {ablation_mode}")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"{ablation_mode}_loss_curve_seed{SEED}.png", dpi=200
    )
    plt.close()
    return model, best_epoch, best_val_loss, checkpoint_path

def completed_modes():
    if not RESULTS_PATH.is_file():
        return set()
    existing = pd.read_csv(RESULTS_PATH)
    if "ablation_mode" not in existing.columns:
        return set()
    return set(existing["ablation_mode"].dropna().astype(str))

def append_result(record):
    if RESULTS_PATH.is_file():
        existing = pd.read_csv(RESULTS_PATH)
        existing = existing[existing["ablation_mode"] != record["ablation_mode"]]
        updated = pd.concat([existing, pd.DataFrame([record])], ignore_index=True)
    else:
        updated = pd.DataFrame([record])
    updated.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")


In [ ]:
combo_df, drug_features, cell_features = load_data()
data_df = preprocess_data(combo_df, drug_features, cell_features)
if data_df.empty:
    raise ValueError("No valid samples remain after preprocessing.")

drug_feat_dim = drug_features.iloc[0].values[2:].astype(np.float32).shape[0]
cell_feat_dim = cell_features.iloc[0].values.astype(np.float32).shape[0]
print(f"Drug feature dimension: {drug_feat_dim}")
print(f"Cell feature dimension: {cell_feat_dim}")
print(f"Valid samples: {len(data_df):,}")

train_df, val_df, test_df, split_summary = grouped_train_val_test_split(
    data_df, train_size=0.70, val_size=0.10, test_size=0.20, random_state=SEED
)
split_summary.to_csv(
    OUTPUT_DIR / f"idb_component_ablation_split_summary_seed{SEED}.csv",
    index=False,
    encoding="utf-8-sig",
)

group_assignments = pd.concat(
    [
        train_df[["group_id"]].drop_duplicates().assign(split="train"),
        val_df[["group_id"]].drop_duplicates().assign(split="validation"),
        test_df[["group_id"]].drop_duplicates().assign(split="test"),
    ],
    ignore_index=True,
)
group_assignments.insert(0, "seed", SEED)
group_assignments.to_csv(
    OUTPUT_DIR / f"idb_component_ablation_group_assignments_seed{SEED}.csv",
    index=False,
    encoding="utf-8-sig",
)

train_dataset = DrugCombinationDataset(train_df, drug_features, cell_features)
val_dataset = DrugCombinationDataset(val_df, drug_features, cell_features)
test_dataset = DrugCombinationDataset(test_df, drug_features, cell_features)

def make_loaders():
    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, val_loader, test_loader

modes_to_run = ["full", *ABLATION_MODES]

already_completed = completed_modes()
print(f"Already completed: {sorted(already_completed)}")

for ablation_mode in modes_to_run:
    if ablation_mode in already_completed:
        print(f"Skipping completed mode: {ablation_mode}")
        continue

    print("\n" + "=" * 100)
    print(f"IDB COMPONENT ABLATION | {ablation_mode} | seed={SEED} | alpha={ALPHA:.2f}")
    print("=" * 100)
    set_global_seed(SEED)
    train_loader, val_loader, test_loader = make_loaders()
    model = DrugCombinationModel(
        drug_feat_dim=drug_feat_dim, cell_feat_dim=cell_feat_dim
    )
    trained_model, best_epoch, best_val_loss, checkpoint_path = train_one_mode(
        model, train_loader, val_loader, val_df, ablation_mode
    )

    test_metrics, test_predictions = evaluate_model(
        trained_model,
        test_loader,
        test_df,
        ablation_mode,
        collect_results=True,
    )
    prediction_path = OUTPUT_DIR / f"{ablation_mode}_test_predictions_seed{SEED}.csv"
    test_predictions.to_csv(prediction_path, index=False, encoding="utf-8-sig")

    record = {
        "seed": SEED,
        "ablation_mode": ablation_mode,
        "alpha": ALPHA,
        "best_epoch": best_epoch,
        "best_validation_mse": best_val_loss,
        **test_metrics,
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "test_samples": len(test_df),
        "train_groups": train_df["group_id"].nunique(),
        "val_groups": val_df["group_id"].nunique(),
        "test_groups": test_df["group_id"].nunique(),
        "source": str(prediction_path),
    }
    append_result(record)

    print("\nIndependent test metrics:")
    for name in ["mse", "rmse", "mae", "pcc", "r2", "c_index"]:
        print(f"{name.upper()}: {test_metrics[name]:.6f}")
    print(f"Saved predictions to: {prediction_path}")

    del model, trained_model, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll requested IDB component-ablation modes are complete.")


In [ ]:
summary = pd.read_csv(RESULTS_PATH)
mode_order = [
    "full_reference",
    "full",
    "no_epsilon",
    "fixed_theta",
    "fixed_theta_no_epsilon",
]
summary["_order"] = summary["ablation_mode"].map(
    {name: index for index, name in enumerate(mode_order)}
).fillna(len(mode_order))
summary = summary.sort_values("_order").drop(columns="_order").reset_index(drop=True)

reference_rows = summary[summary["ablation_mode"].isin(["full_reference", "full"])]
if reference_rows.empty:
    raise ValueError("No full-model reference is available for the comparison.")
reference = reference_rows.iloc[0]
summary["delta_rmse_vs_full"] = summary["rmse"] - float(reference["rmse"])
summary["delta_mae_vs_full"] = summary["mae"] - float(reference["mae"])
summary["delta_pcc_vs_full"] = summary["pcc"] - float(reference["pcc"])
summary["delta_r2_vs_full"] = summary["r2"] - float(reference["r2"])
summary["delta_c_index_vs_full"] = summary["c_index"] - float(reference["c_index"])
summary.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")

display_columns = [
    "ablation_mode",
    "best_epoch",
    "mse",
    "rmse",
    "mae",
    "pcc",
    "r2",
    "c_index",
    "delta_rmse_vs_full",
    "delta_pcc_vs_full",
    "delta_r2_vs_full",
]
print("IDB component-ablation comparison (seed=42, alpha=0.5):")
display(summary[display_columns])
print(f"Saved final summary to: {RESULTS_PATH.resolve()}")
